# IPL Fielding Performance Analysis

This notebook analyzes the fielding contributions of three selected players from an IPL-style fielding dataset. The goal is to quantify fielding impact using the requested scoring formula and present the result in a clear performance summary with charts.

The workbook loads an Excel file, filters to Rohit Sharma, Suryakumar Yadav, and Hardik Pandya, calculates the required action metrics, and compares their overall fielding performance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

excel_path = 'fielding_data.xlsx'
df = pd.read_excel(excel_path)

# Normalize the key fielding fields
df['Player Name'] = df['Player Name'].astype(str)
df['Team'] = df['Team'].astype(str)
df['Pick'] = df['Pick'].astype(str).str.upper()
df['Throw'] = df['Throw'].astype(str).str.upper()
df['Runs'] = pd.to_numeric(df['Runs'], errors='coerce').fillna(0)
df['Position'] = pd.to_numeric(df['Position'], errors='coerce').fillna(0).astype(int)
df['Ball Count'] = pd.to_numeric(df['Ball Count'], errors='coerce').fillna(0).astype(int)
df['Overcount'] = pd.to_numeric(df['Overcount'], errors='coerce').fillna(0).astype(int)

# Ensure the action columns are numeric for aggregation
for col in ['Catch_Success', 'Dropped_Catches', 'Stumping', 'Run_Out', 'Missed_RunOut', 'Direct_Hit']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

selected_players = ['Rohit Sharma', 'Suryakumar Yadav', 'Hardik Pandya']
fielding_data = df[df['Player Name'].isin(selected_players)].copy()
fielding_data.head()

## 1) Aggregate player metrics

We calculate the per-player fielding metrics from the provided columns. The key measures are clean picks, good throws, catches, dropped catches, stumpings, run-outs, missed run-outs, direct hits, and total runs saved/conceded.

In [ ]:
player_summary = (
    fielding_data.groupby('Player Name', as_index=False)
    .agg(
        Position=('Position', lambda s: ', '.join(sorted(map(str, set(s.astype(int).tolist()))))),
        CP=('Pick', lambda s: (s == 'Y').sum()),
        GT=('Throw', lambda s: (s == 'Y').sum()),
        C=('Catch_Success', 'sum'),
        DC=('Dropped_Catches', 'sum'),
        ST=('Stumping', 'sum'),
        RO=('Run_Out', 'sum'),
        MRO=('Missed_RunOut', 'sum'),
        DH=('Direct_Hit', 'sum'),
        RS=('Runs', 'sum')
    )
)

# Performance Score formula
player_summary['PS'] = (
    player_summary['CP'] * 1
    + player_summary['GT'] * 1
    + player_summary['C'] * 3
    - player_summary['DC'] * 3
    + player_summary['ST'] * 3
    + player_summary['RO'] * 3
    - player_summary['MRO'] * 2
    + player_summary['DH'] * 2
    + player_summary['RS']
)

player_summary = player_summary[[
    'Player Name', 'Position', 'CP', 'GT', 'C', 'DC', 'ST', 'RO', 'MRO', 'DH', 'RS', 'PS'
]]

player_summary

## 2) Performance score formula

The score is calculated with the following formula:

$PS = (CP 	imes 1) + (GT 	imes 1) + (C 	imes 3) - (DC 	imes 3) + (ST 	imes 3) + (RO 	imes 3) - (MRO 	imes 2) + (DH 	imes 2) + RS$

Where each metric is aggregated per player before the score is computed.

In [ ]:
# Final summary table for display
summary_table = player_summary.sort_values('PS', ascending=False).reset_index(drop=True)
summary_table

In [ ]:
# 1) Bar chart: Performance Score comparison
plt.figure(figsize=(8, 5))
ax = sns.barplot(data=summary_table, x='Player Name', y='PS', hue='Player Name', dodge=False, palette='viridis', legend=False)
ax.set_title('Performance Score Comparison by Player')
ax.set_xlabel('Player')
ax.set_ylabel('Performance Score (PS)')
for container in ax.containers:
    ax.bar_label(container, padding=3, fmt='%.0f')
plt.tight_layout()
plt.show()

In [ ]:
# 2) Stacked bar: action breakdown by player
stack_df = summary_table[['Player Name', 'CP', 'GT', 'C', 'DC', 'ST', 'RO', 'MRO', 'DH']].melt(id_vars='Player Name', var_name='Action Type', value_name='Value')

plt.figure(figsize=(11, 6))
ax = sns.barplot(data=stack_df, x='Player Name', y='Value', hue='Action Type', palette='Set2')
ax.set_title('Fielding Action Breakdown by Player')
ax.set_xlabel('Player')
ax.set_ylabel('Count')
plt.legend(title='Action Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3) Key insights

The final ranking identifies the best overall fielder and highlights the main reason behind that performance. This summary helps explain the player's fielding impact in simple terms for a short presentation or walkthrough.

In [ ]:
top_player = summary_table.iloc[0]
print(f'Top fielder: {top_player[Player Name]} (PS = {top_player[PS]})')
print('')
for _, row in summary_table.iterrows():
    strongest_area = row[['CP', 'GT', 'C', 'ST', 'RO', 'DH']].idxmax()
    runs_msg = 'saved runs' if row['RS'] >= 0 else 'conceded runs'
    print(f"{row['Player Name']} strongest area: {strongest_area}. This player contributes most through {strongest_area} and has a net {runs_msg} impact of {row['RS']}.")

print('')
print(f"Overall insight: {top_player['Player Name']} leads the fielding chart because they combine strong positional work, accurate throws, and high-impact fielding actions such as catches, run-outs, and direct hits.")